# SOMD 2026 — Mention Substitution Noise Injection
Replaces a mention string with another software mention sampled from the training set,
simulating a context mismatch error where the detector identifies a span as a software
mention but associates it with the wrong software name.

## 0. Imports & Setup

In [1]:
import json
from copy import deepcopy
from tqdm import tqdm
import re
import os
import random

## 1. Load Data

In [2]:
# ── Paths — adjust as needed ──────────────────────────────────────────────────
SUBTASK = "subtask 2"  # change to "subtask 2"

file_to_load = f"../data/SOMD 2026/{SUBTASK}/train_data.jsonl"
with open(file_to_load, 'r') as f:
    train_data = [json.loads(l) for l in f]

with open(f"../data/SOMD 2026/{SUBTASK}/train_labels.json", "r") as f:
    train_labels = json.load(f)

print(f"Loaded {len(train_data):,} mentions")
print(f"Loaded {len(train_labels):,} clusters")
print(f"\nExample mention:")
print(json.dumps(train_data[0], indent=2))

Loaded 2,860 mentions
Loaded 699 clusters

Example mention:
{
  "mention": "NQuery Advisor",
  "mention_id": "1efe624d70",
  "start": 0,
  "end": 14,
  "type": "Application_Usage",
  "docid": "3ca094cf",
  "relations": [
    {
      "mention": "3 . 0",
      "mention_id": "PMC6343886/sentence88/T4",
      "start": 36,
      "end": 41,
      "type": "Version"
    }
  ],
  "sentence": "NQuery Advisor by StatsSols version 3 . 0 was used to calculate sample size ."
}


## 2. Noise Function — Mention Substitution

For each selected mention, we replace its mention string with a **different** software
mention string sampled uniformly from the full training set. The span boundaries
(`start`, `end`) are updated to reflect the new string length. The substitution
string is always sampled from a **different mention** (never the same mention_id),
and we avoid sampling the same surface form to ensure the substitution is genuinely noisy.

This simulates a realistic pipeline error where a mention detection system
correctly identifies a software span but assigns it an incorrect name —
for instance, due to a lookup error or confusion between similar software names.

In [3]:
def inject_mention_substitution_noise(train_data, noise_rate, max_attempts=10):
    """
    Replace mention strings with a different software mention sampled
    from the training set.

    For each selected mention:
    - Sample a replacement string from a different mention in the corpus
    - Update the mention string and span boundaries accordingly
    - The sentence text itself is NOT modified; only the mention span is updated

    Args:
        train_data   : list of mention dicts
        noise_rate   : fraction of mentions to perturb (0.0 to 1.0)
        max_attempts : max resampling attempts to find a different surface form

    Returns:
        Perturbed copy of train_data
    """
    new_train_data = deepcopy(train_data)

    # Build pool of (mention_string, mention_id) pairs for sampling
    mention_pool = [(m['mention'], m['mention_id']) for m in train_data]

    n_to_modify = int(len(new_train_data) * noise_rate)
    indices_to_modify = set(random.sample(range(len(new_train_data)), n_to_modify))

    stats = {'substituted': 0, 'skipped_no_candidate': 0, 'skipped_same_form': 0}

    for idx, mention in enumerate(tqdm(new_train_data, desc="Injecting mention substitution noise")):
        if idx not in indices_to_modify:
            continue

        original_surface = mention['mention'].strip().lower()
        original_id      = mention['mention_id']
        sentence         = mention['sentence']
        start            = mention['start']
        end              = mention['end']

        # Sample a replacement with a different surface form
        replacement = None
        for _ in range(max_attempts):
            candidate_surface, candidate_id = random.choice(mention_pool)
            if (candidate_id != original_id and
                    candidate_surface.strip().lower() != original_surface):
                replacement = candidate_surface.strip()
                break

        if replacement is None:
            stats['skipped_no_candidate'] += 1
            continue

        # Replace the mention span in the sentence and update boundaries
        # We keep the sentence structure intact and only update the annotated span
        new_sentence = sentence[:start] + replacement + sentence[end:]
        new_end      = start + len(replacement)

        mention['mention']  = replacement
        mention['sentence'] = new_sentence
        mention['end']      = new_end
        # start remains unchanged

        stats['substituted'] += 1

    print(f"\nMention substitution noise injection stats:")
    for k, v in stats.items():
        print(f"  {k}: {v}")
    total = stats['substituted']
    print(f"  Total modified: {total}/{len(new_train_data)} ({total/len(new_train_data):.1%})")

    return new_train_data

## 3. Quick Sanity Check

In [4]:
# Run on a tiny sample to verify behaviour before full run
sample = train_data[:20]
noisy_sample = inject_mention_substitution_noise(sample, noise_rate=1.0)

print("\n── Before / After comparison ──")
for orig, noisy in zip(sample[:5], noisy_sample[:5]):
    if orig['mention'] != noisy['mention']:
        print(f"  ORIGINAL : '{orig['mention']}'")
        print(f"  NOISY    : '{noisy['mention']}'")
        print(f"  SENTENCE : ...{noisy['sentence'][max(0,noisy['start']-20):noisy['end']+20]}...")
        print()

Injecting mention substitution noise: 100%|██████████| 20/20 [00:00<00:00, 79663.89it/s]


Mention substitution noise injection stats:
  substituted: 20
  skipped_no_candidate: 0
  skipped_same_form: 0
  Total modified: 20/20 (100.0%)

── Before / After comparison ──
  ORIGINAL : 'NQuery Advisor'
  NOISY    : 'SPSS Statistics'
  SENTENCE : ...SPSS Statistics by StatsSols versio...

  ORIGINAL : 'STATA'
  NOISY    : 'CSPRO'
  SENTENCE : ...CSPRO 12 . 0 by StataCorp...

  ORIGINAL : 'Stata'
  NOISY    : 'Visual C + +'
  SENTENCE : ...s was undertaken in Visual C + + , version 13 . 1 [ ...

  ORIGINAL : 'Statistical Parametric Mapping'
  NOISY    : 'CSPRO'
  SENTENCE : ... and analyzed using CSPRO SPM 8 ( Wellcome De...

  ORIGINAL : 'SPM'
  NOISY    : 'Matlab'
  SENTENCE : ... Parametric Mapping Matlab 8 ( Wellcome Depart...



## 4. Generate Noisy Datasets

In [5]:
noise_rates = [0, 0.25, 0.50, 0.75, 1.0]

output_dir = f"../../SOMD-2026/data/SOMD 2026/{SUBTASK}/noisy_data_substitution"
os.makedirs(output_dir, exist_ok=True)

for noise_rate in noise_rates:
    print(f"\n{'='*50}")
    print(f"Generating noise rate: {noise_rate:.0%}")
    print(f"{'='*50}")

    noisy_data = inject_mention_substitution_noise(train_data, noise_rate=noise_rate)

    subtask_tag = SUBTASK.replace(' ', '_')
    filename = os.path.join(
        output_dir,
        f"noisy_{noise_rate}_train_data_{subtask_tag}_substitution.jsonl"
    )

    with open(filename, 'w') as f:
        for item in noisy_data:
            f.write(json.dumps(item) + '\n')

    print(f"Saved: {filename}")

print("\nAll done.")


Generating noise rate: 0%


Injecting mention substitution noise: 100%|██████████| 2860/2860 [00:00<00:00, 7323387.94it/s]


Mention substitution noise injection stats:
  substituted: 0
  skipped_no_candidate: 0
  skipped_same_form: 0
  Total modified: 0/2860 (0.0%)
Saved: ../../SOMD-2026/data/SOMD 2026/subtask 2/noisy_data_substitution/noisy_0_train_data_subtask_2_substitution.jsonl

Generating noise rate: 25%


Injecting mention substitution noise: 100%|██████████| 2860/2860 [00:00<00:00, 2253985.24it/s]



Mention substitution noise injection stats:
  substituted: 715
  skipped_no_candidate: 0
  skipped_same_form: 0
  Total modified: 715/2860 (25.0%)
Saved: ../../SOMD-2026/data/SOMD 2026/subtask 2/noisy_data_substitution/noisy_0.25_train_data_subtask_2_substitution.jsonl

Generating noise rate: 50%


...


Mention substitution noise injection stats:
  substituted: 1430
  skipped_no_candidate: 0
  skipped_same_form: 0
  Total modified: 1430/2860 (50.0%)
Saved: ../../SOMD-2026/data/SOMD 2026/subtask 2/noisy_data_substitution/noisy_0.5_train_data_subtask_2_substitution.jsonl

Generating noise rate: 75%


Injecting mention substitution noise: 100%|██████████| 2860/2860 [00:00<00:00, 1140276.56it/s]



Mention substitution noise injection stats:
  substituted: 2145
  skipped_no_candidate: 0
  skipped_same_form: 0
  Total modified: 2145/2860 (75.0%)
Saved: ../../SOMD-2026/data/SOMD 2026/subtask 2/noisy_data_substitution/noisy_0.75_train_data_subtask_2_substitution.jsonl

Generating noise rate: 100%


Injecting mention substitution noise: 100%|██████████| 2860/2860 [00:00<00:00, 596801.46it/s]


Mention substitution noise injection stats:
  substituted: 2860
  skipped_no_candidate: 0
  skipped_same_form: 0
  Total modified: 2860/2860 (100.0%)
Saved: ../../SOMD-2026/data/SOMD 2026/subtask 2/noisy_data_substitution/noisy_1.0_train_data_subtask_2_substitution.jsonl

All done.
